<a href="https://colab.research.google.com/github/tetsu19n1101087/python-study-group/blob/main/scr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ライブラリの使い方

In [35]:
import requests
from bs4 import BeautifulSoup

In [36]:
url = "http://www.nikkei.com/"

# url にアクセス
response = requests.get(url)
# エンコーディングを変更（文字化け対策）
response.encoding = response.apparent_encoding

bs = BeautifulSoup(response.text, 'html.parser')

In [37]:
# ページのタイトル
bs.title.string

'日本経済新聞 - ニュース・速報 最新情報'

### 株価の取得

1. [日本経済新聞 > マーケット > 株式](https://www.nikkei.com/markets/worldidx/chart/nk225/) のページにアクセス

2. Chromeで
右クリック → 検証 → 要素 → 右クリック → Copy → Copy selector

3. select_one関数の引数に渡す

In [38]:
url = "https://www.nikkei.com/markets/worldidx/chart/nk225/"

response = requests.get(url)
response.encoding = response.apparent_encoding

bs = BeautifulSoup(response.text, 'html.parser')

In [39]:
bs.title.string

'日経平均株価:リアルタイム推移・最新ニュース - 日本経済新聞'

In [40]:
# htmlのセレクターを指定
bs.select_one("#MAIN_CONTENTS > div:nth-child(6) > div.m-article.economic > div.economic_value > span.economic_value_now.a-fs26").string

'49,001.50'

### サイトからリンクされているページを取得

[千葉大学のホームページ](https://www.chiba-u.ac.jp/) でやってみる

https://www.chiba-u.ac.jp/robots.txt の中身を確認
```
User-agent: *
Disallow: /_mm/
Disallow: /_notes/
Disallow: /_baks/
Disallow: /MMWIP/

User-agent: googlebot
Disallow: *.csi
```

In [41]:
import pandas as pd
from urllib.parse import urljoin, urlparse # urlを編集するライブラリ

url = "https://www.chiba-u.ac.jp/"

# robots.txt から読み取った Disallow パス
disallow_paths = [
    "/_mm/",
    "/_notes/",
    "/_baks/",
    "/MMWIP/",
]

response = requests.get(url)
response.encoding = response.apparent_encoding

bs = BeautifulSoup(response.text, 'html.parser')

In [42]:
links = bs.find_all("a")

results = []

for a in links:
    href = a.get("href")
    title = a.get_text(strip=True)

    # href または title が空なら除外
    if not href or not title:
        continue

    full_url = urljoin(url, href)

    path = urlparse(full_url).path

    # robots.txt で Disallow されているパスを除外
    if any(path.startswith(d) for d in disallow_paths):
        continue

    results.append({
        "title": title,
        "url": full_url,
    })

df = pd.DataFrame(results).drop_duplicates(subset="url").reset_index(drop=True)

In [43]:
df

,title,url
0,寄付をする,https://kikin.chiba-u.ac.jp/
1,日本語,https://www.chiba-u.ac.jp/index.html
2,English,https://www.chiba-u.ac.jp/e/index.html
3,研究,https://www.chiba-u.ac.jp/research-collab/inde...
4,研究最新ニュース,https://www.chiba-u.ac.jp/research-collab/inde...
...,...,...
61,防災,https://www.chiba-u.ac.jp/other/bousai.html
62,情報危機対策チーム(C-csirt),https://www.chiba-u.ac.jp/about/disclosure/sec...
63,採用情報,https://www.chiba-u.ac.jp/about/recruit/index....
64,千葉大学オンラインショップ,https://cu-onlineshop.chiba-u.jp/


### Selenium

In [79]:
!pip3 install --quiet google-colab-selenium
!apt-get install fonts-ipaexfont-gothic fonts-ipaexfont-mincho

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-ipaexfont-gothic is already the newest version (00401-3ubuntu1).
fonts-ipaexfont-mincho is already the newest version (00401-3ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [69]:
import google_colab_selenium as gs
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from PIL import Image
import time
from google.colab import userdata

#環境設定
options = Options()
options.add_argument("--headless")
options.add_argument('--disable-dev-shm-usage')
options.add_argument("--no-sandbox")

mooid_id = userdata.get('moodle_id')
moodle_password = userdata.get('moodle_password')

In [77]:
driver = gs.Chrome(options=options)
driver.get("https://moodle.gs.chiba-u.jp/moodle/login/index.php/")

wait = WebDriverWait(driver, 15)

# ユーザー名
username = wait.until(
    EC.presence_of_element_located((By.ID, "username"))
)
username.send_keys(mooid_id)

# パスワード
password = wait.until(
    EC.presence_of_element_located((By.ID, "password"))
)
password.send_keys(moodle_password)

driver.find_element(By.ID, "loginbtn").click()

<IPython.core.display.Javascript object>

In [ ]:
time.sleep(5)

#Webページのスクリーンショット
driver.save_screenshot("/content/test.png")
driver.quit()

#スクリーンショットの表示
print("\n---Webページのスクショ---\n")
img = Image.open("/content/test.png")
display(img)